In [10]:
%reload_ext autoreload
%autoreload 2

In [11]:
import os
# load env variables from .env file
from dotenv import load_dotenv
load_dotenv("../eval/.env")

True

In [12]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("nasa-impact/nasa_repo_code_benchmark_v0.1", token=os.getenv("HUGGINGFACE_TOKEN"))

In [14]:
dataset_config = {
    "nasa_repo_code_benchmark_v0.1": {
        "path": "nasa-impact/nasa_repo_code_benchmark_v0.1",
        "data_files": [
            "qrels/nasa_science_class_code_docstring_heldout.tsv",
            "qrels/nasa_science_class_code_identifier_heldout.tsv",
            "qrels/nasa_science_function_code_docstring_heldout.tsv",
            "qrels/nasa_science_function_code_identifier_heldout.tsv",
            "qrels/python.tsv",
            "qrels/java.tsv",
            "qrels/javascript.tsv",
            "qrels/c.tsv",
            "qrels/c++.tsv",
            "qrels/fortran.tsv",
            "qrels/matlab.tsv",
        ],
        "data_files_colors": [
            "#1f77b4",  # nasa_science_class_code_docstring_heldout.tsv
            "#ff7f0e",  # nasa_science_class_code_identifier_heldout.tsv
            "#2ca02c",  # nasa_science_function_code_docstring_heldout.tsv
            "#d62728",  # nasa_science_function_code_identifier_heldout.tsv
            "#9467bd",  # python.tsv
            "#8c564b",  # java.tsv
            "#e377c2",  # javascript.tsv
            "#7f7f7f",  # c.tsv
            "#bcbd22",  # c++.tsv
            "#17becf",  # fortran.tsv
            "#aec7e8",  # matlab.tsv
        ],
    },
}

def dataset_getter(
    dataset_name,
    corpus_split="train",
    queries_split="train",
    relevant_docs_split="test",
    data_file=None,
):
    corpus = load_dataset(
        dataset_config[dataset_name]["path"],
        data_files="corpus.jsonl",
        split=corpus_split,
        token=os.environ["HUGGINGFACE_TOKEN"],
    )
    queries = load_dataset(
        dataset_config[dataset_name]["path"],
        data_files="queries.jsonl",
        split=queries_split,
        token=os.environ["HUGGINGFACE_TOKEN"],
    )
    relevant_docs_data = load_dataset(
        dataset_config[dataset_name]["path"],
        split=relevant_docs_split,
        data_files=data_file,
        token=os.environ["HUGGINGFACE_TOKEN"],
    )

    corpus = {row["_id"]: row["text"] for i, row in enumerate(corpus)}
    queries = {row["_id"]: row["text"] for row in queries}
    relevant_docs_data = (
        relevant_docs_data.to_pandas()
        .groupby("query-id")["corpus-id"]
        .apply(set)
        .to_dict()
    )
    relevant_docs_data = {
        str(k): {str(item) for item in v} for k, v in relevant_docs_data.items()
    }

    return corpus, queries, relevant_docs_data

In [22]:
corpus, queries, relevant_docs_data = dataset_getter(
    "nasa_repo_code_benchmark_v0.1",
    corpus_split="train",
    queries_split="train",
    relevant_docs_split="train",
    data_file="qrels/python.tsv",
)

Generating train split: 0 examples [00:00, ? examples/s]

In [23]:
len(corpus)

117950

In [24]:
len(queries)

119720

In [25]:
len(relevant_docs_data)

64110

In [26]:
relevant_docs_data

{'q100000': {'c98381'},
 'q100001': {'c98382'},
 'q100002': {'c98383'},
 'q100003': {'c98384'},
 'q100004': {'c98385'},
 'q100005': {'c98386'},
 'q100006': {'c98387'},
 'q100007': {'c98388'},
 'q100008': {'c98389'},
 'q100009': {'c98390'},
 'q100010': {'c98391'},
 'q100011': {'c98392'},
 'q100012': {'c98393'},
 'q100013': {'c98394'},
 'q100014': {'c98395'},
 'q100015': {'c98396'},
 'q100016': {'c98397'},
 'q100017': {'c98398'},
 'q100018': {'c98399'},
 'q100019': {'c98400'},
 'q100020': {'c98401'},
 'q100021': {'c98402'},
 'q100022': {'c98403'},
 'q100023': {'c98404'},
 'q100024': {'c98405'},
 'q100025': {'c98406'},
 'q100026': {'c98407'},
 'q100027': {'c98408'},
 'q100028': {'c98409'},
 'q100029': {'c98410'},
 'q100030': {'c98411'},
 'q100031': {'c98412'},
 'q100032': {'c98413'},
 'q100033': {'c98414'},
 'q100034': {'c98415'},
 'q100035': {'c98416'},
 'q100036': {'c98417'},
 'q100037': {'c98418'},
 'q100038': {'c98419'},
 'q100039': {'c98420'},
 'q100040': {'c98421'},
 'q100041': {'c9

In [28]:
corpus["c98381"]

'class PerWorkerValuesTypeSpec(type_spec_lib.TypeSpec):\n  def __init__(self, value_spec, descendant_type):\n    assert value_spec\n    self._value_spec = value_spec\n    self._descendant_type = descendant_type\n  def _serialize(self):\n    return (self._value_spec,)\n  @property\n  def value_type(self):\n    return self._descendant_type\n  def most_specific_common_supertype(self, others):\n    raise NotImplementedError(\n        "most_specific_common_supertype is not implemented")\n  @property\n  def _component_specs(self):\n    return self._value_spec\n  def _to_components(self, value):\n    return self._value_spec\n  def _from_components(self, value):\n    return value'